# 阶段 3 笔记本 2：量子优势分析

> **使用本笔记本前建议阅读**：
> - Quantum_Advantage_in_Computational_Chemistry.pdf（IEEE QCE 2025）
> - acs.chemrev.8b00803.pdf 第一节（电子结构问题的计算复杂度）

---

## 战略问题

作为刚进入该领域的研究者，你需要回答：
**在化学中，量子计算机在何处、何时会超越经典计算机？**

本笔记本提供定量工具，帮助你分析这一问题。

---
## 1. 复杂度层级：经典 vs 量子

### 经典方法标度

| 方法 | 标度 | 精度 | 实用上限 |
|--------|---------|----------|-----------------|
| HF | O(N^3) | 无相关 | ~10000 原子 |
| DFT | O(N^3) | 近似相关 | ~1000 原子 |
| MP2 | O(N^5) | 弱相关 | ~500 电子 |
| CCSD | O(N^6) | 较好精度 | ~200 电子 |
| CCSD(T) | O(N^7) | 金标准 | ~100 电子 |
| FCI | O(exp(N)) | 精确 | ~20 轨道 |
| DMRG | O(N^3 chi^3) | 近精确 | ~100 轨道 |

### 量子方法

| 方法 | 电路深度 | 量子比特 | 状态 |
|--------|--------------|--------|--------|
| VQE | 浅 | N_orb | 当前 NISQ |
| SQD | 浅 + 采样 | N_orb | 当前 NISQ |
| QPE | ~O(1/eps^2) * O(N^5) 门 | N_orb | 容错量子计算（2030+） |

### 核心论断
对 FCI 基态的 QPE：**多项式**量子门复杂度  
对比 **指数**经典 FCI 复杂度  
=> 对强相关分子，原则上存在量子 **指数加速**！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 可视化经典“标度墙”
N_electrons = np.arange(2, 80)

# 粗略运算量（非精确，用于直觉）
def ops_hf(N): return N**3
def ops_ccsd(N): return N**6
def ops_ccsdt(N): return N**7
def ops_fci(N): return 2**N  # 很粗：与空间轨道/电子数相关
def ops_dmrg(N, chi=1000): return N * chi**3  # 键维 chi=1000

# 量子 QPE 门数估计（Babbush 等 2018 风格）
# 达到 FCI 量级精度时 T 门 ~ O(N^5 / epsilon^2)
epsilon = 1e-3  # 1 mHa 精度
def ops_qpe(N): return N**5 / epsilon**2  # 数量级

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左：绝对标度
ax = axes[0]
ax.semilogy(N_electrons, [ops_hf(n) for n in N_electrons],   'b-',  lw=2, label='HF O(N^3)')
ax.semilogy(N_electrons, [ops_ccsd(n) for n in N_electrons],  'g-',  lw=2, label='CCSD O(N^6)')
ax.semilogy(N_electrons, [ops_ccsdt(n) for n in N_electrons], 'm-',  lw=2, label='CCSD(T) O(N^7)')
ax.semilogy(N_electrons, [ops_dmrg(n) for n in N_electrons],  'c--', lw=2, label='DMRG chi=1000')
ax.semilogy(N_electrons[:40], [ops_fci(n) for n in N_electrons[:40]], 'r-', lw=2, label='FCI O(exp(N))')
ax.semilogy(N_electrons, [ops_qpe(n) for n in N_electrons],   'k--', lw=2.5, label='QPE O(N^5/eps^2)')

ax.axvline(x=30, color='orange', ls=':', lw=2, label='经典实用上限 ~30 电子')
ax.set_xlabel('体系大小（电子数）', fontsize=12)
ax.set_ylabel('运算量（对数坐标）', fontsize=12)
ax.set_title('经典 vs 量子标度', fontsize=13)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_xlim([2, 70])

# 右：交叉分析
ax = axes[1]
N_range = np.arange(20, 100)

# “量子优势区”：QPE < CCSD(T) 且 FCI 不可行
ccsd_t_ops = np.array([ops_ccsdt(n) for n in N_range])
qpe_ops    = np.array([ops_qpe(n)   for n in N_range])
fci_ops    = np.array([min(ops_fci(n), 1e50) for n in N_range])

ax.semilogy(N_range, ccsd_t_ops, 'm-', lw=2, label='CCSD(T)')
ax.semilogy(N_range, qpe_ops,    'k--',lw=2.5, label='QPE（容错量子）')
ax.semilogy(N_range[:50], fci_ops[:50], 'r-', lw=1.5, label='FCI', alpha=0.7)

# 填充量子优势区域
crossover = N_range[np.where(qpe_ops < ccsd_t_ops)[0]]
if len(crossover) > 0:
    Nc = crossover[0]
    ax.axvline(x=Nc, color='green', ls='--', lw=2, label=f'交叉点 ~{Nc} 电子')
    ax.fill_betweenx([1e10, 1e50], Nc, 100, alpha=0.1, color='green', label='量子优势区')

ax.set_xlabel('体系大小（电子数）', fontsize=12)
ax.set_ylabel('运算量（对数坐标）', fontsize=12)
ax.set_title('量子何时占优？（QPE vs CCSD(T)）', fontsize=13)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_xlim([20, 100])

plt.tight_layout()
plt.savefig('quantum_advantage_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

print('摘自 Quantum_Advantage_in_Computational_Chemistry.pdf 的要点：')
print('FCI 量级精度的量子优势预期出现在 N > 约 50–100 电子时')
print('但：需要容错量子计算（百万级物理量子比特），非 NISQ')
print('对真正经典难解的化学问题，现实时间线约为 10–20 年')

---
## 2. 资源估计：需要多少量子比特与门？

真正的量子优势需要理解 **物理资源成本**。

### 逻辑量子比特 vs 物理量子比特

量子纠错会带来开销：
- **1 个逻辑量子比特** 约需 **~1000 个物理量子比特**（当前估计）
- 100 逻辑量子比特的 FeMoco 类计算约需 ~100,000 物理量子比特
- 当前代表水平：IBM Eagle/Heron = 127–133 物理量子比特
- 量子化学目标：百万级物理量子比特

### QPE 的门数（Babbush 等 2018）

对 N 电子、M 轨道、精度 epsilon 的分子：
$$\text{T-gates} \approx \frac{\lambda}{\epsilon} \cdot \text{circuit overhead}$$

其中 $\lambda = \sum_{pq} |h_{pq}| + \frac{1}{2}\sum_{pqrs} |g_{pqrs}|$（哈密顿量的 1-范数）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 基准分子的资源估计
# 数据来自 Babbush 等 (2018) Nature Communications、Lee 等 (2021)

benchmark_molecules = [
    # (名称, 逻辑量子比特数, T 门数, 备注)
    ('H2 (STO-3G)',         4,      1e3,    '玩具模型，现已可做'),
    ('N2 (cc-pVDZ)',        20,     1e8,    'N2 三键，NISQ 时代目标'),
    ('Fe2S2 (active)',      40,     1e10,   '小型 Fe-S 团簇'),
    ('FeMoco (CAS)',        110,    4e10,   'Reiher 等 2017，标志性估计'),
    ('Cytochrome P450',     200,    1e12,   '工业催化目标'),
    ('Perovskite (TiO2)',   500,    1e14,   '你的非均相催化方向！'),
]

# 物理量子比特开销（表面码，物理错误率 p=0.1%）
def physical_qubits(n_logical, overhead=1000):
    return n_logical * overhead

# 时间估计（假设逻辑门时钟 1 MHz）
def runtime_hours(T_gates, clock_MHz=1):
    return T_gates / (clock_MHz * 1e6 * 3600)

print(f'{"分子":<22}{"逻辑比特":<14}{"物理比特":<15}{"T 门":<12}{"运行时间":<12}{"说明"}')
print('-'*90)
for mol, nq, tg, note in benchmark_molecules:
    phys_q = physical_qubits(nq)
    rt = runtime_hours(tg)
    if rt < 1:
        rt_str = f'{rt*60:.0f} 分钟'
    elif rt < 24:
        rt_str = f'{rt:.1f} 小时'
    elif rt < 8760:
        rt_str = f'{rt/24:.0f} 天'
    else:
        rt_str = f'{rt/8760:.0f} 年'
    print(f'{mol:<22}{nq:<14}{phys_q:<15}{tg:<12.1e}{rt_str:<12}{note}')

print()
print('假设：表面码开销 1000 倍，逻辑时钟 1 MHz')
print('当前 IBM 代表水平：约 133 物理量子比特（2024）')
print('路线图：IBM 计划约 2033 年达 >100,000 量子比特')

# 标度图
fig, ax = plt.subplots(figsize=(10, 5))
names = [m[0].split('(')[0].strip() for m in benchmark_molecules]
log_qubits = [m[1] for m in benchmark_molecules]
t_gates = [np.log10(m[2]) for m in benchmark_molecules]

colors = ['green', 'blue', 'orange', 'red', 'purple', 'darkred']
scatter = ax.scatter(log_qubits, t_gates, s=150, c=colors, zorder=5)
for i, (name, x, y) in enumerate(zip(names, log_qubits, t_gates)):
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.axhline(y=10, color='green', ls='--', lw=1.5, label='T 门=10^10（约 2030 可行？）')
ax.axhline(y=13, color='orange', ls='--', lw=1.5, label='T 门=10^13（约 2040 可行？）')
ax.axvline(x=133, color='gray', ls=':', lw=1.5, label='当前 IBM（约 133 逻辑等效）')
ax.set_xlabel('逻辑量子比特', fontsize=12)
ax.set_ylabel('log10(T 门数)', fontsize=12)
ax.set_title('量子化学目标的资源需求', fontsize=13)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('resource_requirements.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. NISQ 量子优势：近期现实评估

容错 QPE 尚需 10–20 年，但特定化学问题上的 **NISQ 量子优势** 可能更早出现。

### 什么可能带来 NISQ 量子优势？

1. **强相关体系，使各类经典近似失效**
   - 但 DMRG 往往在经典侧也能处理

2. **采样问题**（SQD/QSCI）
   - 量子硬件天然可对高维分布采样
   - 适用于：组态空间探索、蒙特卡洛类方法

3. **量子模拟**（非优化）
   - 直接模拟量子动力学
   - 适用于：光化学、质子转移、量子核效应

4. **混合 QM/QC 方法**
   - 仅用量子计算机处理 10–50 量子比特的「难」子空间
   - 其余由经典 DFT/嵌入处理
   - 这是 **近期最现实的路径！**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 技术就绪时间线分析
# 依据：Quantum_Advantage_in_Computational_Chemistry.pdf
#       Babbush 等 2021 PRX Quantum
#       IBM Quantum Development Roadmap 2024

years = np.arange(2024, 2040)

# 物理量子比特数外推（类摩尔定律）
# IBM：1000（2023）-> 2033 年约 100,000（约每 3 年 10 倍）
phys_qubits = 1000 * (10 ** ((years - 2023) / 3))

# 错误率改善：0.1%（2023）-> 0.01%（2030）-> 0.001%（2035）
error_rates = 0.001 * (10 ** (max(0, 2030 - years) / 3.5))
error_rates = np.clip(error_rates, 1e-4, 1e-2)

# 逻辑量子比特估计（逻辑 = 物理 / 开销；随错误率改善开销下降）
# 0.1% 时开销 ~1000，0.01% ~100，0.001% ~10
overhead = 10 ** (3 * error_rates / 0.001)
overhead = np.clip(overhead, 10, 2000)
logical_qubits = phys_qubits / overhead

# 化学里程碑
milestones = [
    (4,   'H2 FCI (STO-3G)', 'green'),
    (20,  'N2 FCI (cc-pVDZ)', 'blue'),
    (40,  'Fe2S2 活性空间', 'orange'),
    (110, 'FeMoco (Reiher 估计)', 'red'),
    (300, '工业催化剂（进取目标）', 'purple'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.semilogy(years, phys_qubits, 'b-', lw=2, label='物理量子比特（外推）')
ax.semilogy(years, logical_qubits, 'r-', lw=2, label='逻辑量子比特（估计）')

for nq, name, color in milestones:
    ax.axhline(y=nq, color=color, ls='--', lw=1.2, alpha=0.8, label=f'{name} ({nq} 比特)')

ax.set_xlabel('年份', fontsize=12)
ax.set_ylabel('量子比特数（对数坐标）', fontsize=12)
ax.set_title('量子硬件路线图 vs 化学里程碑', fontsize=12)
ax.legend(fontsize=8, loc='lower right'); ax.grid(True, alpha=0.3)

# 研究机会图：当前何处易发表
ax = axes[1]
categories = ['VQE\n算法\n设计', 'SQD/QSCI\n实际\n应用',
              '化学中的\n误差\n缓解', '力场的\nQML',
              '量子\n嵌入\n(QM+QC)', 'QPE\n资源\n估计']
# 坐标轴：5 年影响力 vs 当前可发表性
impact   = [0.6, 0.85, 0.70, 0.90, 0.80, 0.50]
feasibility = [0.90, 0.85, 0.80, 0.95, 0.75, 0.60]
sizes = [300, 350, 280, 380, 320, 220]
colors = ['steelblue','forestgreen','darkorange','firebrick','mediumpurple','gray']

for i, (cat, imp, feas, sz, col) in enumerate(zip(categories, impact, feasibility, sizes, colors)):
    ax.scatter(feas, imp, s=sz, c=col, alpha=0.8, zorder=5)
    ax.annotate(cat, (feas, imp), ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points', fontsize=8, fontweight='bold')

ax.set_xlabel('当前可行性（可发表结果）', fontsize=12)
ax.set_ylabel('5 年影响力（引用/领域重要性）', fontsize=12)
ax.set_title('研究机会图（结合你的背景）', fontsize=12)
ax.set_xlim([0.5, 1.05]); ax.set_ylim([0.4, 1.05])
ax.grid(True, alpha=0.3)

# 标出最佳机会区
rect = plt.Rectangle((0.80, 0.78), 0.24, 0.25, fill=False, edgecolor='green', lw=2, ls='--')
ax.add_patch(rect)
ax.text(0.82, 1.02, '甜点区！', color='green', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('research_opportunity_map.png', dpi=150, bbox_inches='tight')
plt.show()

print('结合你背景的战略甜点区：')
print('1. 力场 QML：可行性最高 + 影响力高')
print('   - 用量子核方法扩展 ReaxFF QML 工作')
print('   - 用 VQE 生成优于 CCSD(T) 的训练数据')
print()
print('2. 非均相催化的量子嵌入（QM+QC）：')
print('   - 体相用 DFT/周期 + 活性位用 VQE')
print('   - 你的 DFT-多相背景是差异化优势')
print()
print('3. 强相关催化剂的 SQD（现在即可发表）：')
print('   - IBM 开源 qiskit-addon-sqd')
print('   - 单原子催化剂是理想试例')

---
## 4. 阅读指南：Quantum_Advantage_in_Computational_Chemistry.pdf

阅读这篇 IEEE QCE 2025 论文时，可围绕下列问题：

### 第一节（引言）
- 他们如何区分「量子优势」与「量子霸权」？
- 为何选基态能量作为基准问题？

### 第二节（经典方法）
- 他们如何估计经典 CCSD(T) 的交叉点？
- DMRG 在分析中起什么作用？

### 第三–四节（量子方法与资源估计）
- 他们认为哪种量子算法最省资源？
- λ（1-范数）如何影响 T 门数？
- 他们估计量子计算机何时能处理 FeMoco？

### 第五节（结论）
- 「最乐观」时间线是什么？
- 他们指出的主要瓶颈有哪些？

### 你的批判性总结：
读完后回答：**这对你的 3–5 年研究计划意味着什么？**
- 哪些计算在今天 NISQ 硬件上可行？
- 哪些需要容错量子计算，因而 >10 年？
- 你更可能在算法侧还是实验侧贡献？

In [ ]:
import numpy as np

# 小结：常见化学问题的量子优势分类

problems = [
    # (问题, 经典难度, 量子侧状态, 时间线, 与你的相关性)
    ('H2, H2O 基态',     '易（CCSD(T)）',   '暂无',         '现在',   '仅作基准'),
    ('N2 三键',           '难（MRCI）',       'NISQ 部分',     '2025',  '试例'),
    ('Fe 活性位（SAC）',     '很难',         'NISQ 有希望',   '2026',  '你的重点！'),
    ('FeMoco 固氮酶',       '难解',       '需容错 QC',     '2030+', '长期目标'),
    ('钙钛矿表面反应',   'DFT 不准',    '嵌入 QC/QM',  '2027',  '你的重点！'),
    ('激发态动力学',   'TDDFT 不准',  'VQE+qEOM',        '2026',  '与 OLED 相关'),
    ('量子核效应',  'PI-MD 昂贵',   '量子模拟','2028', 'MD 背景'),
    ('力场训练',     'CCSD(T) 昂贵', 'VQE 数据生成',    '2025',  '与 ReaxFF 衔接！'),
]

print('化学问题的量子优势评估')
print('='*90)
print(f'{"问题":<30}{"经典侧":<20}{"量子状态":<20}{"时间线":<10}{"与你相关"}')
print('-'*90)
for p in problems:
    print(f'{p[0]:<30}{p[1]:<20}{p[2]:<20}{p[3]:<10}{p[4]}')

print()
print('研究策略要点：')
print('>>> 阶段 1（现在，1–2 年）：Fe SAC 的 VQE/SQD + 力场 QML 改进')
print('>>> 阶段 2（3–5 年）：      表面反应的量子嵌入')
print('>>> 阶段 3（5–10 年）：     硬件成熟后的 FeMoco/固氮酶')

---
## 5. 值得跟踪的 arXiv 文献

为这些主题设置提醒以保持更新：

### 常用 arXiv 检索
- `quant-ph` + 关键词：VQE, SQD, QSCI, quantum chemistry, error mitigation
- `cond-mat.str-el`：强相关体系、DMRG+量子
- `physics.chem-ph`：量子化学方法

### 近期必读（2023 后）
1. Robledo-Moreno 等 (2024) arXiv:2405.05068 — Nature Chemistry 上的 SQD
2. Kim 等 (2023) Nature — IBM 127 量子比特误差缓解
3. Arrazola 等 (2022) Nature — 真实硬件上的 VQE 基准
4. Chan 等 (2024) — 量子嵌入综述
5. 你文件夹中的：2508.02578（SKQD）与 2512.13657（iQCC）

### 跟踪工具
- https://arxiv-sanity-lite.com/
- https://www.semanticscholar.org/（引用提醒）
- IBM Research 博客：https://research.ibm.com/blog

---
## 小结：你的研究定位

```
经典化学（DFT/MD/ReaxFF）  ←──────────────────────────────────────────────────────┐
         ↓                                                                                   │
         ↓（你的博士背景）                                                                   │
         ↓                                                                                   │
强相关下 DFT 失效：                                                                          │
  - 过渡金属氧化物、SAC、FeMoco                                                              │
  - 含自由基的过渡态                                                                        │
  - 键断裂/形成                                                                             │
         ↓                                                                                   │
  量子计算填补缺口  ────────── 你独特的桥梁位置 ────────────────  ┘
  活性位用 VQE/SQD +
  体相用 DFT 嵌入 +
  动力学用 QML/ReaxFF +
  实现工具用 Qiskit
```

**下一步**：阶段 4 笔记本 — 将以上内容落到你的具体研究方向上。